# 1. Funkcja Weierstrassa

## Wprowadzenie

Funkcja Weierstrassa jest klasycznym przykładem funkcji **ciągłej, która nie jest różniczkowalna w żadnym punkcie**. Została przedstawiona przez Karla Weierstrassa w 1872 roku.

### Definicja

$$W(x) = \sum_{n=0}^{\infty} a^n \cos(b^n \pi x)$$

gdzie:
- $0 < a < 1$ - współczynnik tłumienia amplitudy
- $b > 1$ - współczynnik częstotliwości (zazwyczaj liczba nieparzysta)
- Warunek $ab > 1$ gwarantuje nieróżniczkowalność

### Wymiar fraktalny wykresu

Teoretyczny wymiar Minkowskiego wykresu funkcji Weierstrassa:

$$D = 2 + \frac{\log a}{\log b}$$

Ponieważ $0 < a < 1$, mamy $\log a < 0$, więc $1 < D < 2$.

In [ ]:
# Importy
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

from src.weierstrass import weierstrass_function, theoretical_dimension, weierstrass_derivative_approx
from src.box_counting import function_to_points, box_counting_dimension

# Konfiguracja wykresów
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Moduły załadowane pomyślnie!")

## 1.1 Wizualizacja funkcji Weierstrassa

Zbadajmy jak funkcja Weierstrassa wygląda dla różnych parametrów.

In [ ]:
# Podstawowa wizualizacja
x = np.linspace(0, 2, 10000)

# Standardowe parametry
a, b = 0.5, 3
y = weierstrass_function(x, a=a, b=b, n_terms=50)

plt.figure(figsize=(14, 5))
plt.plot(x, y, 'b-', linewidth=0.5)
plt.xlabel('x')
plt.ylabel('W(x)')
plt.title(f'Funkcja Weierstrassa (a={a}, b={b})')
plt.tight_layout()
plt.savefig('../output/figures/weierstrass_basic.png', dpi=150)
plt.show()

print(f"Teoretyczny wymiar fraktalny: D = {theoretical_dimension(a, b):.4f}")

## 1.2 Wpływ parametrów a i b

Zbadajmy jak zmieniają się właściwości funkcji przy różnych wartościach parametrów.

In [ ]:
# Różne wartości parametru a (przy stałym b=3)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
a_values = [0.3, 0.5, 0.7, 0.8, 0.9, 0.95]
b = 3

for ax, a in zip(axes.flat, a_values):
    y = weierstrass_function(x, a=a, b=b, n_terms=50)
    dim = theoretical_dimension(a, b)
    
    ax.plot(x, y, 'b-', linewidth=0.3)
    ax.set_title(f'a={a}, b={b}\nD = {dim:.3f}')
    ax.set_xlabel('x')
    ax.set_ylabel('W(x)')

plt.suptitle('Wpływ parametru a na funkcję Weierstrassa', fontsize=14)
plt.tight_layout()
plt.savefig('../output/figures/weierstrass_param_a.png', dpi=150)
plt.show()

In [ ]:
# Różne wartości parametru b (przy stałym a=0.5)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
b_values = [3, 5, 7, 9, 11, 15]
a = 0.5

for ax, b in zip(axes.flat, b_values):
    y = weierstrass_function(x, a=a, b=b, n_terms=50)
    dim = theoretical_dimension(a, b)
    
    ax.plot(x, y, 'b-', linewidth=0.3)
    ax.set_title(f'a={a}, b={b}\nD = {dim:.3f}')
    ax.set_xlabel('x')
    ax.set_ylabel('W(x)')

plt.suptitle('Wpływ parametru b na funkcję Weierstrassa', fontsize=14)
plt.tight_layout()
plt.savefig('../output/figures/weierstrass_param_b.png', dpi=150)
plt.show()

## 1.3 Samopodobieństwo funkcji Weierstrassa

Kluczową właściwością fraktali jest **samopodobieństwo** - struktura wygląda podobnie na różnych skalach. Zbadajmy to przez przybliżanie wykresu.

In [ ]:
# Demonstracja samopodobieństwa przez przybliżanie
a, b = 0.5, 3
center = 1.0
zoom_levels = [2.0, 0.5, 0.1, 0.02]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, width in zip(axes, zoom_levels):
    x_min, x_max = center - width/2, center + width/2
    x_zoom = np.linspace(x_min, x_max, 5000)
    y_zoom = weierstrass_function(x_zoom, a=a, b=b, n_terms=100)
    
    ax.plot(x_zoom, y_zoom, 'b-', linewidth=0.5)
    ax.set_title(f'Szerokość okna: {width}')
    ax.set_xlabel('x')
    ax.set_ylabel('W(x)')

plt.suptitle(f'Samopodobieństwo funkcji Weierstrassa (a={a}, b={b})', fontsize=14)
plt.tight_layout()
plt.savefig('../output/figures/weierstrass_zoom.png', dpi=150)
plt.show()

## 1.4 Nieróżniczkowalność funkcji Weierstrassa

Funkcja Weierstrassa jest ciągła, ale nie jest różniczkowalna w żadnym punkcie. Zbadamy to przez obliczenie przybliżenia pochodnej dla różnych wartości h.

In [ ]:
# Przybliżenie pochodnej dla różnych h
a, b = 0.5, 3
x_point = 1.0
h_values = [1e-2, 1e-4, 1e-6, 1e-8, 1e-10]

print("Przybliżenie pochodnej w punkcie x = 1.0:")
print("-" * 40)
print(f"{'h':<15} {'f\'(x) ≈ (f(x+h)-f(x))/h':<20}")
print("-" * 40)

derivatives = []
for h in h_values:
    deriv = weierstrass_derivative_approx(np.array([x_point]), a=a, b=b, h=h)[0]
    derivatives.append(deriv)
    print(f"{h:<15.0e} {deriv:<20.6f}")

print("-" * 40)
print("\nWnioski: Wartości NIE zbiegają do żadnej granicy - funkcja jest nieróżniczkowalna!")

In [ ]:
# Wizualizacja przybliżenia pochodnej na przedziale
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_range = np.linspace(0, 2, 1000)
h_demo = [1e-2, 1e-4, 1e-6]

for ax, h in zip(axes, h_demo):
    deriv = weierstrass_derivative_approx(x_range, a=0.5, b=3, h=h)
    ax.plot(x_range, deriv, 'r-', linewidth=0.5)
    ax.set_title(f'Przybliżenie pochodnej (h = {h:.0e})')
    ax.set_xlabel('x')
    ax.set_ylabel("W'(x) ≈")
    ax.set_ylim(-50, 50)

plt.suptitle('Niezbieżność przybliżenia pochodnej', fontsize=14)
plt.tight_layout()
plt.savefig('../output/figures/weierstrass_derivative.png', dpi=150)
plt.show()

## 1.5 Obliczanie wymiaru Minkowskiego metodą box-counting

Porównajmy teoretyczny wymiar fraktalny z wymiarem obliczonym numerycznie metodą box-counting.

In [ ]:
# Obliczenie wymiaru dla standardowych parametrów
a, b = 0.5, 3
n_points = 20000

# Konwersja funkcji na punkty
points = function_to_points(weierstrass_function, x_range=(0, 2), n_points=n_points, a=a, b=b, n_terms=50)

# Obliczenie wymiaru
dim_numerical, r2, details = box_counting_dimension(points, epsilon_min=0.001, epsilon_max=0.1, n_epsilons=25)
dim_theoretical = theoretical_dimension(a, b)

print(f"Parametry: a = {a}, b = {b}")
print(f"Wymiar teoretyczny:  D = {dim_theoretical:.4f}")
print(f"Wymiar numeryczny:   D = {dim_numerical:.4f}")
print(f"Błąd względny:       {abs(dim_numerical - dim_theoretical) / dim_theoretical * 100:.2f}%")
print(f"R² dopasowania:      {r2:.4f}")

In [ ]:
# Wizualizacja analizy box-counting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lewy wykres - funkcja
x = np.linspace(0, 2, 10000)
y = weierstrass_function(x, a=a, b=b)
axes[0].plot(x, y, 'b-', linewidth=0.3)
axes[0].set_xlabel('x')
axes[0].set_ylabel('W(x)')
axes[0].set_title(f'Funkcja Weierstrassa (a={a}, b={b})')

# Prawy wykres - regresja box-counting
axes[1].scatter(details['log_inv_eps'], details['log_counts'], s=50, c='blue', label='Dane')
axes[1].plot(details['log_inv_eps'], 
             details['slope'] * details['log_inv_eps'] + details['intercept'],
             'r-', linewidth=2, label=f'Regresja: D = {dim_numerical:.4f}')
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('log(1/ε)')
axes[1].set_ylabel('log(N(ε))')
axes[1].set_title(f'Box-counting (teoretyczne D = {dim_theoretical:.4f})')
axes[1].legend()

plt.tight_layout()
plt.savefig('../output/figures/weierstrass_box_counting.png', dpi=150)
plt.show()

## 1.6 Analiza zależności wymiaru od parametrów

Zbadajmy jak wymiar fraktalny zmienia się w zależności od parametrów a i b.

In [ ]:
# Analiza wymiaru vs parametr a (przy stałym b=3)
a_values = np.linspace(0.3, 0.9, 10)
b_fixed = 3

dims_numerical_a = []
dims_theoretical_a = []

print("Obliczanie wymiarów dla różnych wartości a...")
for i, a in enumerate(a_values):
    # Wymiar teoretyczny
    dim_theo = theoretical_dimension(a, b_fixed)
    dims_theoretical_a.append(dim_theo)
    
    # Wymiar numeryczny
    points = function_to_points(weierstrass_function, (0, 2), 15000, a=a, b=b_fixed, n_terms=50)
    dim_num, _, _ = box_counting_dimension(points, epsilon_min=0.002, epsilon_max=0.1, n_epsilons=20)
    dims_numerical_a.append(dim_num)
    
    print(f"  a = {a:.2f}: D_theo = {dim_theo:.4f}, D_num = {dim_num:.4f}")

dims_numerical_a = np.array(dims_numerical_a)
dims_theoretical_a = np.array(dims_theoretical_a)

In [ ]:
# Wykres porównawczy
plt.figure(figsize=(10, 6))
plt.plot(a_values, dims_theoretical_a, 'b-o', linewidth=2, markersize=8, label='Wymiar teoretyczny')
plt.plot(a_values, dims_numerical_a, 'r--s', linewidth=2, markersize=8, label='Wymiar numeryczny (box-counting)')
plt.xlabel('Parametr a')
plt.ylabel('Wymiar fraktalny D')
plt.title(f'Zależność wymiaru od parametru a (b = {b_fixed})')
plt.legend()
plt.ylim(1, 2)
plt.grid(True, alpha=0.3)
plt.savefig('../output/figures/weierstrass_dimension_vs_a.png', dpi=150)
plt.show()

# Błąd średni
mean_error = np.mean(np.abs(dims_numerical_a - dims_theoretical_a))
print(f"\nŚredni błąd bezwzględny: {mean_error:.4f}")

In [ ]:
# Analiza wymiaru vs parametr b (przy stałym a=0.5)
b_values = np.array([3, 5, 7, 9, 11, 13, 15, 17, 19, 21])
a_fixed = 0.5

dims_numerical_b = []
dims_theoretical_b = []

print("Obliczanie wymiarów dla różnych wartości b...")
for i, b in enumerate(b_values):
    # Wymiar teoretyczny
    dim_theo = theoretical_dimension(a_fixed, b)
    dims_theoretical_b.append(dim_theo)
    
    # Wymiar numeryczny
    points = function_to_points(weierstrass_function, (0, 2), 15000, a=a_fixed, b=b, n_terms=50)
    dim_num, _, _ = box_counting_dimension(points, epsilon_min=0.002, epsilon_max=0.1, n_epsilons=20)
    dims_numerical_b.append(dim_num)
    
    print(f"  b = {b}: D_theo = {dim_theo:.4f}, D_num = {dim_num:.4f}")

dims_numerical_b = np.array(dims_numerical_b)
dims_theoretical_b = np.array(dims_theoretical_b)

In [ ]:
# Wykres porównawczy
plt.figure(figsize=(10, 6))
plt.plot(b_values, dims_theoretical_b, 'b-o', linewidth=2, markersize=8, label='Wymiar teoretyczny')
plt.plot(b_values, dims_numerical_b, 'r--s', linewidth=2, markersize=8, label='Wymiar numeryczny (box-counting)')
plt.xlabel('Parametr b')
plt.ylabel('Wymiar fraktalny D')
plt.title(f'Zależność wymiaru od parametru b (a = {a_fixed})')
plt.legend()
plt.ylim(1, 1.8)
plt.grid(True, alpha=0.3)
plt.savefig('../output/figures/weierstrass_dimension_vs_b.png', dpi=150)
plt.show()

# Błąd średni
mean_error = np.mean(np.abs(dims_numerical_b - dims_theoretical_b))
print(f"\nŚredni błąd bezwzględny: {mean_error:.4f}")

## 1.7 Podsumowanie

### Kluczowe wnioski:

1. **Funkcja Weierstrassa** jest ciągła, ale nigdzie nie różniczkowalna - jest to pierwszy historyczny przykład takiej funkcji.

2. **Parametr a** kontroluje "szorstkość" wykresu:
   - Większe a → większy wymiar fraktalny → bardziej "wypełniony" wykres
   - Mniejsze a → wymiar bliższy 1 → wykres bardziej "gładki"

3. **Parametr b** kontroluje częstotliwość oscylacji:
   - Większe b → szybsze oscylacje → mniejszy wymiar fraktalny

4. **Wymiar fraktalny** obliczony metodą box-counting jest zgodny z wzorem teoretycznym:
   $$D = 2 + \frac{\log a}{\log b}$$

5. **Samopodobieństwo** jest wyraźnie widoczne przy przybliżaniu wykresu.

In [ ]:
# Tabela podsumowująca dla obu analiz
import pandas as pd

# Tabela dla zmiennego a (b=3)
results_a = {
    'a': a_values,
    'b': [b_fixed] * len(a_values),
    'D_teoretyczny': dims_theoretical_a,
    'D_numeryczny': dims_numerical_a,
    'Błąd (%)': np.abs(dims_numerical_a - dims_theoretical_a) / dims_theoretical_a * 100
}

df_a = pd.DataFrame(results_a)
df_a.to_csv('../output/data/weierstrass_dimensions_vs_a.csv', index=False)

# Tabela dla zmiennego b (a=0.5)
results_b = {
    'a': [a_fixed] * len(b_values),
    'b': b_values,
    'D_teoretyczny': dims_theoretical_b,
    'D_numeryczny': dims_numerical_b,
    'Błąd (%)': np.abs(dims_numerical_b - dims_theoretical_b) / dims_theoretical_b * 100
}

df_b = pd.DataFrame(results_b)
df_b.to_csv('../output/data/weierstrass_dimensions_vs_b.csv', index=False)

print("Wyniki zapisane do:")
print("  - output/data/weierstrass_dimensions_vs_a.csv")
print("  - output/data/weierstrass_dimensions_vs_b.csv")

print("\n" + "="*70)
print("Tabela dla zmiennego a (b=3):")
print("="*70)
print(df_a.to_string(index=False))

print("\n" + "="*70)
print("Tabela dla zmiennego b (a=0.5):")
print("="*70)
print(df_b.to_string(index=False))